### Notebook for current source density analysis

##### Imports

In [ ]:
import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from probeinterface import read_probeinterface
from probeinterface.plotting import plot_probe
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

##### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

raw_data_dir_env = "RAW_DATA_DIR_DUAL"
data_fname_env = "SLEEP_TEST_DATA_FNAME_HPC"
probe_config_env = "PROBE_CONFIG_PATH_HPC"
derivatives_dir_env = "DERIVATIVES_DIR_DUAL"

raw_data_dir = Path(os.getenv(raw_data_dir_env))
data_fname = Path(os.getenv(data_fname_env))
probe_config_path = Path(os.getenv(probe_config_env))
derivatives_dir = Path(os.getenv(derivatives_dir_env))
full_data_path = os.path.join(raw_data_dir, data_fname)
config_path = os.path.join(repo_root, "config/preprocessing.yml")
lfp_output_dir = os.path.join(derivatives_dir, "lfp")

# Load preprocessing config
with open(Path(config_path)) as f:
    config = yaml.safe_load(f)
    
sample_rate = config['recording']['sample_rate']
n_channels = config['recording']['n_channels']
dtype = config['recording']['dtype']
freq_min = config['lfp']['bandpass_filter']['freq_min']
freq_max = config['lfp']['bandpass_filter']['freq_max']
reference = config['lfp']['common_reference']['reference']
operator = config['lfp']['common_reference']['operator']
resample_rate = config['lfp']['resample_rate']

In [ ]:
recording = se.read_binary(full_data_path, sample_rate, dtype, n_channels)

In [ ]:
data_fname.stem

##### Read probe layout from config file

In [ ]:
probe_group = read_probeinterface(probe_config_path)
probe = probe_group.probes[0]

active_mask = probe.device_channel_indices != -1
active_probe = probe.get_slice(active_mask)

fig, ax = plt.subplots(figsize=(4, 12))
colors = ["blue" if ch != -1 else "white" for ch in probe.device_channel_indices]
plot_probe(probe, ax=ax, contacts_colors=colors, with_device_index=False)
plt.tight_layout()

# active_probe has 384 contacts, all mapped 1:1 to recording channels.
# The full probe has 5120 contacts (384 active); copy_metadata in preprocessors silently
# drops it when contact count != channel count, causing the "no Probe attached" warning.
recording_with_probe = recording.set_probe(active_probe)

##### Bad channel detection and interpolation

In [ ]:
# NP 2.0 standard gain (0.01950 µV/LSB, same as SpikeGLX AP-band default).
# Verify against your acquisition system metadata if available.
NP2_GAIN_TO_UV = 0.01950

# coherence+psd requests traces in µV internally; wrap the recording to supply gain metadata
recording_scaled = si.scale(recording_with_probe, gain=NP2_GAIN_TO_UV, dtype="float32")
recording_scaled.set_channel_gains(0.976259); recording_scaled.set_channel_offsets(0)

bad_channel_ids, channel_labels = sp.detect_bad_channels(recording_scaled)

unique_labels, counts = np.unique(channel_labels, return_counts=True)
print("Channel summary:", dict(zip(unique_labels, counts)))
print(f"Bad channels ({len(bad_channel_ids)}):", bad_channel_ids)

# Interpolate bad channels from their neighbours to preserve uniform electrode spacing for CSD
recording_interpolated = sp.interpolate_bad_channels(recording_scaled, bad_channel_ids)

##### LFP Preprocessing

In [ ]:
import warnings

# Preprocessing pipeline (lazy chain - no data is loaded yet)
recording_filtered = sp.bandpass_filter(recording_interpolated, freq_min=freq_min, freq_max=freq_max)
recording_cmr = sp.common_reference(recording_filtered, reference=reference, operator=operator)
recording_lfp = sp.resample(recording_cmr, resample_rate=resample_rate)

# Save to disk (skip if folder already exists)
lfp_save_path = Path(lfp_output_dir) / 'lfp_preprocessed' / data_fname.stem
if not lfp_save_path.exists():
    print('Saving preprocessed LFP to disk...')
    # Worker processes reconstruct the chain via dump_to_dict, which doesn't carry probe
    # metadata from in-place set_probe calls. This doesn't affect computation: global
    # median CMR doesn't use electrode positions. Suppress the cosmetic warning.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="There is no Probe attached", category=UserWarning)
        recording_lfp.save(folder=lfp_save_path, n_jobs=4, chunk_duration='1s')
    print(f'Saved to {lfp_save_path}')
else:
    print(f'Found existing LFP recording: {lfp_save_path}')

##### Load Preprocessed LFP

In [ ]:
lfp_save_path = Path(lfp_output_dir) / 'lfp_preprocessed' / data_fname.stem
recording_lfp = si.load_extractor(lfp_save_path)
print(recording_lfp)

##### Shank Decomposition

In [ ]:
# Build per-shank channel maps sorted by depth
# active_probe has 384 contacts: 96 per shank, 2 columns x 48 depth levels each

contact_positions = np.array(active_probe.contact_positions)   # (384, 2) um
shank_ids_arr = active_probe.shank_ids                         # (384,)
device_ch_idx = active_probe.device_channel_indices            # (384,) -> rec channel index

shank_meta = {}
for shank_id in np.unique(shank_ids_arr):
    on_shank = shank_ids_arr == shank_id
    positions = contact_positions[on_shank]            # (96, 2)
    ch_idx = device_ch_idx[on_shank]                   # (96,) recording channel indices

    unique_depths = np.sort(np.unique(positions[:, 1]))  # (48,) um, ascending
    depth_groups = [
        ch_idx[positions[:, 1] == d]                     # ~2 channels per depth level
        for d in unique_depths
    ]
    shank_meta[shank_id] = {'depth_groups': depth_groups, 'depths_um': unique_depths}

# Electrode spacing from probe geometry (um -> m)
sample_depths = list(shank_meta.values())[0]['depths_um']
electrode_spacing_m = float(np.mean(np.diff(sample_depths))) * 1e-6

print(f'Shanks: {list(shank_meta.keys())}')
print(f'Depth levels per shank: {len(sample_depths)}, spacing: {electrode_spacing_m * 1e6:.1f} um')

##### Extract LFP per Shank and Compute CSD

In [ ]:
# --- Time window (set to None for full recording) ---
t_start_s = 0.0
t_end_s   = 15

start_frame = int(t_start_s * resample_rate) if t_start_s is not None else 0
end_frame = int(t_end_s   * resample_rate) if t_end_s   is not None else recording_lfp.get_num_frames()

# Load traces for the window: (n_samples, 384)
traces = recording_lfp.get_traces(start_frame=start_frame, end_frame=end_frame).astype(np.float32)


def extract_shank_lfp(traces, depth_groups):
    # Average columns at each depth -> (n_depths, n_samples)
    return np.stack([traces[:, grp].mean(axis=1) for grp in depth_groups], axis=0)


def compute_csd_fd(lfp, spacing_m):
    # Finite-difference CSD: -d^2V/dz^2, shape (n_depths - 2, n_samples)
    return -np.diff(lfp, n=2, axis=0) / spacing_m ** 2


# --- iCSD option (requires elephant + quantities) ---
# from elephant.current_source_density import estimate_csd
# import quantities as pq
# def compute_csd_icsd(lfp, spacing_m, method='StandardCSD'):
#     coords = (np.arange(lfp.shape[0]) * spacing_m) * pq.m
#     return estimate_csd(lfp * pq.uV, coords, method=method)


shank_lfp = {sid: extract_shank_lfp(traces, meta['depth_groups']) for sid, meta in shank_meta.items()}
shank_csd = {sid: compute_csd_fd(lfp, electrode_spacing_m) for sid, lfp in shank_lfp.items()}

print({sid: csd.shape for sid, csd in shank_csd.items()})

##### Visualize CSD

In [ ]:
time_s   = np.arange(start_frame, end_frame) / resample_rate
n_shanks = len(shank_csd)

fig, axes = plt.subplots(1, n_shanks, figsize=(4 * n_shanks, 8), sharey=False)
if n_shanks == 1:
    axes = [axes]

for ax, (shank_id, csd) in zip(axes, shank_csd.items()):
    depths = shank_meta[shank_id]['depths_um'][1:-1]  # inner electrodes only
    vmax   = np.percentile(np.abs(csd), 99)

    im = ax.imshow(
        csd,
        aspect='auto',
        origin='lower',
        extent=[time_s[0], time_s[-1], depths[0], depths[-1]],
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_title(f'Shank {shank_id}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Depth (um)')
    plt.colorbar(im, ax=ax, shrink=0.6, label='CSD (V/m^2)')

plt.suptitle('Current Source Density', fontsize=14)
plt.tight_layout()
plt.show()

##### Epoching (optional)

In [ ]:
# --- Epoch around events and average before computing CSD ---
# Uncomment and adapt when event timestamps are available.
#
# event_times_s    = np.array([...])   # 1-D array of event times in seconds
# pre_s,  post_s   = 0.5, 0.5          # window around each event
# pre_frames       = int(pre_s  * resample_rate)
# post_frames      = int(post_s * resample_rate)
#
# def epoch_lfp(lfp, event_frames, pre_frames, post_frames):
#     # Baseline-subtract and average aligned epochs -> (n_depths, n_window_samples)
#     valid  = (event_frames >= pre_frames) & (event_frames + post_frames <= lfp.shape[1])
#     epochs = np.stack(
#         [lfp[:, f - pre_frames : f + post_frames] for f in event_frames[valid]], axis=0
#     )
#     epochs -= epochs.mean(axis=2, keepdims=True)  # per-epoch baseline subtract
#     return epochs.mean(axis=0)
#
# event_frames      = (event_times_s * resample_rate).astype(int)
# shank_lfp_epoched = {sid: epoch_lfp(lfp, event_frames, pre_frames, post_frames)
#                      for sid, lfp in shank_lfp.items()}
# shank_csd_epoched = {sid: compute_csd_fd(lfp, electrode_spacing_m)
#                      for sid, lfp in shank_lfp_epoched.items()}
#
# Plot shank_csd_epoched the same way as above, with:
# time_s = np.linspace(-pre_s, post_s, pre_frames + post_frames)